# 🎨 The Policy Briefing in 40 Minutes
### Unit 1 Activity — Colour Theory in Data Visualization (UE24CS342AA9)

---

**11:00 AM.** You're a junior analyst at **GlobalIndex Insights**. In 40 minutes you're
presenting well-being trends to a policy think-tank. Your Creative Director, Meera,
left these notes:

> *"Every chart in this draft misuses color somewhere -- wrong scale type, no
> accessible palette, a rainbow colormap, you name it. Fix each one using the right
> color-mapping principle. I've marked what's broken with a `# TODO`."*

Same dataset the team has been using: the World Happiness Report 2015.

## ⏱️ Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import TwoSlopeNorm
import time

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

df = pd.read_csv("ppt6_2015.csv")
mission_start = time.time()
print(f"Loaded {len(df)} rows. 40 minutes on the clock. Go.")
df.head()

In [ ]:
# Run any time to check your remaining budget
elapsed_min = (time.time() - mission_start) / 60
remaining = max(0, 40 - elapsed_min)
print(f"Elapsed: {elapsed_min:0.1f} min   |   Remaining: {remaining:0.1f} min")

---
## Task 1 — Pick the Right Aesthetic *(~4 min)*

Meera's note: *"We need to compare Economy (GDP per Capita) across the top 10
countries. Right now it's plotted with only position -- add color as a SECOND aesthetic
encoding the same variable, so the redundancy makes the comparison unmistakable."*

In [ ]:
top10 = df.nsmallest(10, "Happiness Rank")

fig, ax = plt.subplots(figsize=(7, 5))

values = top10["Economy (GDP per Capita)"]
bar_colors = plt.cm.Blues(plt.Normalize(values.min(), values.max())(values))

ax.barh(top10["Country"], values, color=bar_colors)
ax.set_title("Top 10 Countries: Economy (GDP per Capita)")
ax.set_xlabel("Economy (GDP per Capita)")
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

---
## Task 2 — Build the Region x Factor Heatmap *(~5 min)*

Meera's note: *"I want the exact 'average value in a colored matrix' chart from the
lecture -- rows are regions, columns are well-being factors, cell color is the average
score. Use a sequential colormap since there's no natural zero-midpoint here."*

In [ ]:
factor_cols = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
              "Freedom", "Trust (Government Corruption)", "Generosity"]

region_matrix = df.groupby("Region")[factor_cols].mean()

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(region_matrix.values, cmap="YlGnBu", aspect="auto")

ax.set_xticks(range(len(factor_cols)))
ax.set_xticklabels(factor_cols, rotation=35, ha="right")
ax.set_yticks(range(len(region_matrix.index)))
ax.set_yticklabels(region_matrix.index)
ax.set_title("Average Happiness Factors by Region")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
## Task 3 — Fix the Qualitative Scale *(~5 min)*

Meera's note: *"This regional comparison chart currently colors every country the same
color -- there's no way to see the regional pattern. Give each region its own
consistent, equally-weighted color."*

In [ ]:
ordered = df.sort_values("Happiness Score", ascending=True)
regions = ordered["Region"].unique()

palette = plt.cm.tab10(np.linspace(0,1,len(regions)))
color_map = {r: palette[i] for i,r in enumerate(regions)}

bar_colors = ordered["Region"].map(color_map)

fig, ax = plt.subplots(figsize=(7, 11))
ax.barh(ordered["Country"], ordered["Happiness Score"], color=bar_colors)
ax.tick_params(axis="y", labelsize=5)
ax.set_title("All Countries by Happiness Score")
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

---
## Task 4 — Build a Diverging Scale *(~5 min)*

Meera's note: *"I want Freedom scores shown as deviation from the GLOBAL AVERAGE, not
raw values -- above-average countries should read as one color family, below-average as
another, meeting at a neutral midpoint. Use a diverging colormap centered at zero."*

In [ ]:
freedom_dev = df["Freedom"] - df["Freedom"].mean()

df_dev = df.assign(freedom_dev=freedom_dev)
sample = pd.concat([df_dev.nlargest(10, "freedom_dev"), df_dev.nsmallest(10, "freedom_dev")])
sample = sample.sort_values("freedom_dev")

norm = TwoSlopeNorm(vmin=sample["freedom_dev"].min(), vcenter=0, vmax=sample["freedom_dev"].max())

fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(sample["Country"], sample["freedom_dev"], color=plt.cm.RdBu_r(norm(sample["freedom_dev"])))
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Freedom Deviation from Global Mean")
plt.tight_layout()
plt.show()

---
## Task 5 — Build an Accent Scale *(~6 min)*

Meera's note: *"Within Southern Asia, I want ONE chart that mutes every country to gray
EXCEPT the country with the highest Generosity score and the country with the lowest --
those two are the story. Use vivid, contrasting accent colors for just those two."*

In [ ]:
region_df = df[df["Region"] == "Southern Asia"].sort_values("Generosity")

lowest_country = region_df.iloc[0]["Country"]
highest_country = region_df.iloc[-1]["Country"]

colors = ["#d9d9d9"] * len(region_df)
colors[list(region_df["Country"]).index(lowest_country)] = "#D55E00"
colors[list(region_df["Country"]).index(highest_country)] = "#0072B2"

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(region_df["Country"], region_df["Generosity"], color=colors)
ax.set_title("Southern Asia: Generosity (Accent Scale)")
plt.tight_layout()
plt.show()

---
## Task 6 — Kill the Rainbow Colormap *(~5 min)*

Meera's note: *"Whoever built this heatmap used 'jet'. Convert it to grayscale first so
you can SEE why that's a problem, then replace it with a perceptually uniform
colormap."*

In [ ]:
matrix = df.groupby("Region")[factor_cols].mean().values

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].imshow(matrix, cmap="jet", aspect="auto")
axes[0].set_title("Current: 'jet' colormap", fontsize=10)

jet_rgba = plt.cm.jet(plt.Normalize(matrix.min(), matrix.max())(matrix))
gray_jet = 0.2126*jet_rgba[...,0] + 0.7152*jet_rgba[...,1] + 0.0722*jet_rgba[...,2]
axes[1].imshow(gray_jet, cmap="gray", aspect="auto")
axes[1].set_title("'jet' converted to grayscale")

axes[2].imshow(matrix, cmap="viridis", aspect="auto")
axes[2].set_title("Improved: 'viridis' colormap")

for a in axes:
    a.set_xticks([])
    a.set_yticks([])

plt.tight_layout()
plt.show()

## Reflection

1. **Sequential** color scales are best for ordered numeric values because light-to-dark progression matches magnitude.

2. **Qualitative** palettes give each category an equal visual weight, making regions easier to distinguish.

3. **Diverging** scales are useful when data has a meaningful midpoint, such as deviations from the global mean.

4. `viridis` is more accessible than `jet` because it has perceptually uniform brightness and remains readable in grayscale.